# Benchmarking Alignment

Functional alignment with demo benchmarking: source shuffle, within-target-bin shuffle, zero-value baselines, prediction table evaluation, and ranking correlation / top-k overlap.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
import scanpy as sc
from sklearn.linear_model import RidgeCV

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.utils.config import load_yaml_config
from model.utils.reproducibility import seed_everything
from model.utils.device import get_device
from model.models import Forecaster, ForecasterConfig
from model.training.checkpointing import load_checkpoint
from model.data.trajectory_pairs import PrepareTrajectoryData, TrajectoryDataset
from model.data.preprocessing import prepare_clusters
from model.analysis import (
    make_global_source_shuffle_items,
    make_within_target_bin_shuffle_items,
    make_zero_value_items,
    predict_items,
    evaluate_prediction_table,
    ranking_correlation_table,
    topk_overlap_table,
    aggregate_robustness_summary,
    target_bins_for_items,
    target_bin_mean_from_reference,
    target_values_from_items,
    score_latent_time_correlation,
    score_expression_abundance_variance,
    evaluate_gene_ranking_against_panel,
    random_tf_baseline,
    parse_label,
)

DEVICE = get_device()
print('DEVICE:', DEVICE)


In [ ]:
# ===== Parameters =====
SMOKE_MODE = True            # False -> fuller benchmarking
SEED = 42
seed_everything(SEED)

# Data subsetting
MAX_ITEMS = 100 if SMOKE_MODE else None   # None = use all heldout
BATCH_SIZE = 32 if SMOKE_MODE else 128

# Shuffle seeds
N_SHUFFLE_SEEDS = 1 if SMOKE_MODE else 5

# Ranking / top-k
TOP_K_VALUES = [5, 10, 20] if SMOKE_MODE else [5, 10, 20, 50]

OUTDIR = ROOT / 'artifacts' / 'notebooks' / 'benchmarking'
OUTDIR.mkdir(parents=True, exist_ok=True)

GEN_CFG = load_yaml_config(ROOT / 'configs' / 'forecasting.yaml')
adata_path = ROOT / GEN_CFG['data']['h5ad_path']
gen_ckpt_path = ROOT / GEN_CFG['training']['checkpoint_path']

print('adata_path:', adata_path)
print('gen_ckpt_path:', gen_ckpt_path)
print('SMOKE_MODE:', SMOKE_MODE)
print('MAX_ITEMS:', MAX_ITEMS)
print('N_SHUFFLE_SEEDS:', N_SHUFFLE_SEEDS)

In [ ]:
# Load data and model
adata = sc.read_h5ad(adata_path)
adata = prepare_clusters(adata)
print(adata)

prep = PrepareTrajectoryData(
    h5ad_path=str(adata_path),
    config={'model_params': GEN_CFG['model']},
    subset_col=GEN_CFG['data'].get('subset_col', 'trajectory_class'),
    subset_values=tuple(GEN_CFG['data'].get('subset_values', ['PV'])),
    time_col=GEN_CFG['data'].get('time_col'),
    cluster_col=GEN_CFG['data'].get('cluster_col'),
    n_bins=int(GEN_CFG['data'].get('n_bins', 120)),
    allowed_offsets=tuple(GEN_CFG['data'].get('allowed_offsets', [1,2,3,4,5,6])),
    base_max_dist=float(GEN_CFG['data'].get('base_max_dist', 12.0)),
    dist_alpha=float(GEN_CFG['data'].get('dist_alpha', 1.0)),
    allowed_cross_steps=tuple(GEN_CFG['data'].get('allowed_cross_steps', [1,2])),
    k_intra=int(GEN_CFG['data'].get('k_intra', 1)),
    k_cross=int(GEN_CFG['data'].get('k_cross', 2)),
    val_split=float(GEN_CFG['data'].get('val_split', 0.2)),
    heldout_split=float(GEN_CFG['data'].get('heldout_split', 0.1)),
    pair_diagnostics=False,
)

all_items = prep.heldout_data
n_total = len(all_items)
items = all_items if MAX_ITEMS is None else all_items[:MAX_ITEMS]
print(f'Using {len(items)} / {n_total} heldout items')

# Load forecaster
f_model = Forecaster(ForecasterConfig.from_dict(GEN_CFG['model'])).to(DEVICE)
f_model, _ = load_checkpoint(f_model, gen_ckpt_path, device=DEVICE)
f_model.eval()
print('Forecaster loaded.')


## 1) Full Model (Reference)

In [ ]:
# Predict with full (unaltered) model as reference
print('Running full-model predictions ...')
full_preds, full_trues, full_sources, full_meta = predict_items(
    f_model, items, device=DEVICE, batch_size=BATCH_SIZE
)
print(f'full_preds: {full_preds.shape}, full_trues: {full_trues.shape}')

## 2) Source Shuffle Baseline

In [ ]:
shuffle_rows = []

for shuffle_seed in range(N_SHUFFLE_SEEDS):
    print(f' Source shuffle seed={shuffle_seed} ...')
    shuffled_items = make_global_source_shuffle_items(items, seed=shuffle_seed)
    shuf_preds, shuf_trues, shuf_sources, shuf_meta = predict_items(
        f_model, shuffled_items, device=DEVICE, batch_size=BATCH_SIZE
    )
    row = evaluate_prediction_table(
        preds=shuf_preds,
        trues=shuf_trues,
        original_sources=full_sources,
        meta=shuf_meta,
        adata=prep.adata,
        condition='source_shuffle',
        shuffle_seed=shuffle_seed,
        full_preds=full_preds,
        altered_sources=shuf_sources,
    )
    shuffle_rows.append(row)

source_shuffle_df = pd.DataFrame(shuffle_rows)
display(source_shuffle_df)

## 3) Within-Target-Bin Shuffle Baseline

In [ ]:
wbin_rows = []
n_bins_val = int(GEN_CFG['data'].get('n_bins', 120))
time_col_val = GEN_CFG['data'].get('time_col')

for shuffle_seed in range(N_SHUFFLE_SEEDS):
    print(f' Within-bin shuffle seed={shuffle_seed} ...')
    wb_items = make_within_target_bin_shuffle_items(
        items, adata=prep.adata, seed=shuffle_seed,
        n_bins=n_bins_val, time_col=time_col_val
    )
    wb_preds, wb_trues, wb_sources, wb_meta = predict_items(
        f_model, wb_items, device=DEVICE, batch_size=BATCH_SIZE
    )
    row = evaluate_prediction_table(
        preds=wb_preds,
        trues=wb_trues,
        original_sources=full_sources,
        meta=wb_meta,
        adata=prep.adata,
        condition='within_target_bin_shuffle',
        shuffle_seed=shuffle_seed,
        full_preds=full_preds,
        altered_sources=wb_sources,
    )
    wbin_rows.append(row)

within_bin_shuffle_df = pd.DataFrame(wbin_rows)
display(within_bin_shuffle_df)

## 4) Zero-Value Baseline

In [ ]:
print('Running zero-value baseline ...')
zero_items = make_zero_value_items(items)
zero_preds, zero_trues, zero_sources, zero_meta = predict_items(
    f_model, zero_items, device=DEVICE, batch_size=BATCH_SIZE
)

zero_row = evaluate_prediction_table(
    preds=zero_preds,
    trues=zero_trues,
    original_sources=full_sources,
    meta=zero_meta,
    adata=prep.adata,
    condition='zero_value',
    full_preds=full_preds,
    altered_sources=zero_sources,
)

zero_value_df = pd.DataFrame([zero_row])
display(zero_value_df)

## 5) Full Model (Prediction Table)

In [ ]:
# Evaluate full model itself as a condition
full_row = evaluate_prediction_table(
    preds=full_preds,
    trues=full_trues,
    original_sources=full_sources,
    meta=full_meta,
    adata=prep.adata,
    condition='full_model',
)

full_model_df = pd.DataFrame([full_row])

# Combine all conditions into a single evaluation table
eval_frames = [full_model_df]
if not source_shuffle_df.empty:
    eval_frames.append(source_shuffle_df)
if not within_bin_shuffle_df.empty:
    eval_frames.append(within_bin_shuffle_df)
if not zero_value_df.empty:
    eval_frames.append(zero_value_df)

benchmark_table = pd.concat(eval_frames, ignore_index=True)
print('\n=== Combined Benchmark Table ===')
display(benchmark_table)

## 6) Ranking Correlation & Top-K Overlap

In [ ]:
# Build a per-condition aggregate summary for ranking comparison
# Use transition_direction_cosine_mean as the primary ranking score
if len(benchmark_table) >= 2:
    rank_candidates = benchmark_table[['condition', 'transition_direction_cosine_mean']].dropna().copy()
    rank_candidates = rank_candidates.rename(columns={'transition_direction_cosine_mean': 'score'})

    if len(rank_candidates) >= 2:
        # Ranking correlation (Spearman across conditions)
        # Note: ranking_correlation_table expects a 'score_col' and 'setting_col'
        # We structure the data accordingly
        print('Ranking table:')
        display(rank_candidates)

        # For top-k overlap, compute pairwise if we have settings/conditions
        # This is primarily meaningful with perturbation panels; here we log the scores
        try:
            corr_df, piv = ranking_correlation_table(
                df=rank_candidates,
                score_col='score',
                setting_col='condition',
            )
            print('\nRanking correlation:')
            display(corr_df)
            corr_df.to_csv(OUTDIR / 'ranking_correlation.csv', index=False)
            print('Saved:', OUTDIR / 'ranking_correlation.csv')
        except Exception as e:
            print(f'Ranking correlation skipped: {e}')

        # Top-k overlap
        try:
            for k in TOP_K_VALUES:
                topk_df = topk_overlap_table(
                    df=rank_candidates,
                    score_col='score',
                    setting_col='condition',
                    k=k,
                )
                if not topk_df.empty:
                    print(f'\nTop-{k} overlap:')
                    display(topk_df)
                    topk_df.to_csv(OUTDIR / f'top{k}_overlap.csv', index=False)
                    print(f'Saved: {OUTDIR / f"top{k}_overlap.csv"}')
        except Exception as e:
            print(f'Top-k overlap skipped: {e}')

        # Aggregate robustness summary
        try:
            robust_summary = aggregate_robustness_summary(corr_df, topk_df if 'topk_df' in dir() else pd.DataFrame())
            if robust_summary:
                with open(OUTDIR / 'robustness_summary.json', 'w') as f:
                    json.dump(robust_summary, f, indent=2)
                print('Saved:', OUTDIR / 'robustness_summary.json')
        except Exception as e:
            print(f'Robustness summary skipped: {e}')
else:
    print('Skipped ranking/top-k: insufficient conditions.')

## 7) Target-Bin Mean Baseline

In [ ]:
# Target-bin mean baseline: predict the mean expression of training cells
# in the same target time bin, ignoring source cell identity.
print('Building target-bin mean baseline ...')

# Use training data as reference for bin means
_n_train = min(len(prep.train_data), max(MAX_ITEMS * 3, 300)) if MAX_ITEMS else len(prep.train_data)
_train_items = prep.train_data[:_n_train]
print(f'Using {_n_train} training items for bin-mean reference')

# Compute per-bin mean from training, project to heldout bins
_train_targets = target_values_from_items(_train_items)
_ref_bins = target_bins_for_items(_train_items, adata=prep.adata, n_bins=n_bins_val, time_col=time_col_val)
_query_bins = target_bins_for_items(items, adata=prep.adata, n_bins=n_bins_val, time_col=time_col_val)

bin_mean_preds = target_bin_mean_from_reference(
    ref_trues=_train_targets,
    ref_target_bins=_ref_bins,
    query_target_bins=_query_bins,
)

bin_mean_row = evaluate_prediction_table(
    preds=bin_mean_preds,
    trues=full_trues,
    original_sources=full_sources,
    meta=full_meta,
    adata=prep.adata,
    condition='target_bin_mean',
)

bin_mean_df = pd.DataFrame([bin_mean_row])
print('Target-bin mean baseline:')
display(bin_mean_df)


## 8) Ridge Regression Baseline

In [ ]:
# Ridge regression baseline: fit multi-output Ridge using source expression
# as features to predict target expression.
print('Building Ridge regression baseline ...')

def _build_source_features(items_list):
    features = []
    for item in items_list:
        fv = item['full_input_val']
        if hasattr(fv, 'numpy'):
            fv = fv.numpy()
        features.append(np.asarray(fv, dtype=np.float32).ravel())
    return np.vstack(features)

# Build train/test feature matrices using the same training subset
X_train = _build_source_features(_train_items)
y_train_r = target_values_from_items(_train_items)
X_test = _build_source_features(items)

print(f'Fitting RidgeCV on {X_train.shape[0]} samples x {X_train.shape[1]} features ...')
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0], store_cv_values=True)
ridge.fit(X_train, y_train_r)
print(f'Best alpha: {ridge.alpha_:.1f}')

ridge_preds = ridge.predict(X_test)

ridge_row = evaluate_prediction_table(
    preds=ridge_preds,
    trues=full_trues,
    original_sources=full_sources,
    meta=full_meta,
    adata=prep.adata,
    condition='ridge',
)

ridge_df = pd.DataFrame([ridge_row])
print('Ridge regression baseline:')
display(ridge_df)


## 9) Conventional Ranking Comparison

In [ ]:
# Build conventional gene rankings and compare against perturbation panel.
# Rankings: logit_diff (from 01), cluster_tf (from 01),
#   latent_time_corr, expr_abundance, expr_variance (in-silico from adata).
print('=== Conventional Ranking Comparison ===')
TOP_K = 8

rankings = {}  # {method: DataFrame[Gene, score]}

# 1) Logit differential (from 01_classifier_analysis output)
logit_path = ROOT / 'artifacts' / 'notebooks' / 'classifier' / 'global_logit_impact_summary.csv'
if logit_path.exists():
    df = pd.read_csv(logit_path)
    g_col = 'Gene' if 'Gene' in df.columns else df.columns[0]
    s_col = 'Mean_Impact' if 'Mean_Impact' in df.columns else df.columns[1]
    rankings['logit_diff'] = df[[g_col, s_col]].rename(columns={g_col: 'Gene', s_col: 'score'}).dropna()
    print(f'Loaded logit_diff: {len(rankings["logit_diff"])} genes')
else:
    print('[WARN] logit_diff missing — run 01_classifier_analysis first')

# 2) Cluster TF impact (from 01 output)
tf_path = ROOT / 'artifacts' / 'notebooks' / 'classifier' / 'branch_tf_impact_cluster_summary.csv'
if tf_path.exists():
    df = pd.read_csv(tf_path)
    for col_cand in ['Abs_Mean_IG_to_PV', 'Mean_IG_to_PV']:
        if col_cand in df.columns:
            rankings['cluster_tf'] = df[['Gene', col_cand]].rename(columns={col_cand: 'score'}).dropna()
            break
    if 'cluster_tf' in rankings:
        print(f'Loaded cluster_tf: {len(rankings["cluster_tf"])} genes')
else:
    print('[WARN] cluster_tf missing — run 01_classifier_analysis first')

# 3-5) In-silico rankings from adata (always available)
X_dense = adata.X.toarray() if hasattr(adata.X, 'toarray') else np.asarray(adata.X, dtype=np.float32)
gene_names_list = [str(g) for g in adata.var_names]

lt_corr = score_latent_time_correlation(adata, X_dense, time_col=GEN_CFG['data'].get('time_col'))
rankings['latent_time_corr'] = pd.DataFrame({'Gene': gene_names_list, 'score': np.abs(np.nan_to_num(lt_corr, nan=0.0))})

abundance = np.asarray(X_dense.mean(axis=0)).ravel()
rankings['expr_abundance'] = pd.DataFrame({'Gene': gene_names_list, 'score': np.nan_to_num(abundance, nan=0.0)})

variance = np.asarray(X_dense.var(axis=0)).ravel()
rankings['expr_variance'] = pd.DataFrame({'Gene': gene_names_list, 'score': np.nan_to_num(variance, nan=0.0)})

print(f'Built {len(rankings)} ranking methods: {list(rankings.keys())}')


In [ ]:
# Score each ranking against perturbation panel (from 04 output)
pert_scores_path = ROOT / 'artifacts' / 'notebooks' / 'perturbation' / 'multi_readout_scores.csv'

if pert_scores_path.exists():
    pert_df = pd.read_csv(pert_scores_path)
    # Extract gene from label: "MYT1L_OE" -> "MYT1L"
    pert_df['Gene'] = pert_df['label'].apply(lambda x: parse_label(str(x))[0])
    # Per gene: best multi_readout_score across OE/KD variants
    gene_scores = pert_df.groupby('Gene', observed=True)['multi_readout_score'].max().reset_index()
    print(f'Loaded perturbation gene scores: {len(gene_scores)} genes')

    # Evaluate each ranking
    comparison_rows = []
    for method, rank_df in rankings.items():
        if rank_df.empty:
            continue
        top_genes = rank_df.dropna().sort_values('score', ascending=False).head(TOP_K)['Gene'].tolist()
        matched = gene_scores[gene_scores['Gene'].isin(top_genes)]
        comparison_rows.append({
            'method': method,
            'top_k': TOP_K,
            'n_matched': len(matched),
            'mean_perturbation_score': float(matched['perturbation_score'].mean()) if not matched.empty else float('nan'),
            'top_genes': ','.join(top_genes[:5]),
        })

    comparison_df = pd.DataFrame(comparison_rows).sort_values('mean_perturbation_score', ascending=False)

    # Random TF baseline
    n_rand = 50 if SMOKE_MODE else 500
    all_gene_pool = gene_scores['Gene'].unique().tolist()
    rng = np.random.default_rng(SEED)
    rand_scores = []
    for _ in range(n_rand):
        rand_genes = rng.choice(all_gene_pool, size=min(TOP_K, len(all_gene_pool)), replace=False).tolist()
        matched_r = gene_scores[gene_scores['Gene'].isin(rand_genes)]
        rand_scores.append(float(matched_r['perturbation_score'].mean()) if not matched_r.empty else float('nan'))

    rand_mean = float(np.nanmean(rand_scores))
    rand_q95 = float(np.nanpercentile(rand_scores, 95))
    print(f'\nRandom TF baseline (n={n_rand}): mean={rand_mean:.4f}, q95={rand_q95:.4f}')

    comparison_df['random_baseline_mean'] = rand_mean
    comparison_df['random_baseline_q95'] = rand_q95
    comparison_df['above_random_q95'] = comparison_df['mean_perturbation_score'] > rand_q95

    print(f'\nConventional ranking vs perturbation panel (top {TOP_K}):')
    display(comparison_df)
else:
    comparison_df = pd.DataFrame()
    print('[INFO] Perturbation scores not found — run 04_perturbation_analysis first to enable ranking comparison')
    print(f'  Expected: {pert_scores_path}')


In [ ]:
# Rebuild combined benchmark table with all conditions
eval_frames = [full_model_df]
if not source_shuffle_df.empty:
    eval_frames.append(source_shuffle_df)
if not within_bin_shuffle_df.empty:
    eval_frames.append(within_bin_shuffle_df)
if not zero_value_df.empty:
    eval_frames.append(zero_value_df)
if not bin_mean_df.empty:
    eval_frames.append(bin_mean_df)
if not ridge_df.empty:
    eval_frames.append(ridge_df)

benchmark_table = pd.concat(eval_frames, ignore_index=True)
print('\n=== Updated Combined Benchmark Table ===')
display(benchmark_table)

# Save all benchmark results
benchmark_table.to_csv(OUTDIR / 'benchmark_evaluation_table.csv', index=False)
print('Saved:', OUTDIR / 'benchmark_evaluation_table.csv')

if not source_shuffle_df.empty:
    source_shuffle_df.to_csv(OUTDIR / 'source_shuffle_baseline.csv', index=False)
if not within_bin_shuffle_df.empty:
    within_bin_shuffle_df.to_csv(OUTDIR / 'within_bin_shuffle_baseline.csv', index=False)
if not zero_value_df.empty:
    zero_value_df.to_csv(OUTDIR / 'zero_value_baseline.csv', index=False)
if not bin_mean_df.empty:
    bin_mean_df.to_csv(OUTDIR / 'target_bin_mean_baseline.csv', index=False)
if not ridge_df.empty:
    ridge_df.to_csv(OUTDIR / 'ridge_baseline.csv', index=False)
if not comparison_df.empty:
    comparison_df.to_csv(OUTDIR / 'conventional_ranking_comparison.csv', index=False)

# Summary JSON
benchmark_summary = {
    'n_heldout_items': int(len(items)),
    'conditions': benchmark_table['condition'].unique().tolist(),
    'full_model_cosine_mean': float(full_row.get('transition_direction_cosine_mean', float('nan'))),
    'full_model_raw_mse': float(full_row.get('raw_mse', float('nan'))),
    'ridge_best_alpha': float(ridge.alpha_) if 'ridge' in dir() else None,
}
with open(OUTDIR / 'benchmark_summary.json', 'w') as f:
    json.dump(benchmark_summary, f, indent=2)

print(f'\n=== All benchmarking outputs saved to {OUTDIR} ===')
